In [ ]:
import requests
import time
import pandas as pd
from typing import Optional, Dict, List
from id_converter import url_id_to_numeric, numeric_to_url_id


In [ ]:
def fetch_event(event_id: int, timeout: int = 10) -> Optional[Dict]:
    """
    Fetch raw event data from the API.
    
    Returns:
        dict if valid event
        None if invalid / not found / error
    """
    str_event_id = numeric_to_url_id(event_id)
    url = f"https://results.advancedeventsystems.com/api/event/{str_event_id}"
    
    try:
        response = requests.get(url, timeout=timeout)
        
        if response.status_code != 200:
            return None
        
        data = response.json()
        
        if not data or 'EventId' not in data:
            print("No data")
            return None
        
        return data
    
    except requests.RequestException:
        return None
        

In [ ]:
def parse_event(data: Dict) -> Dict:
    """
    Extract relevant fields from event data.
    """
    event_id = data.get('EventId')
    event_name = data.get('name', '')
    occurred = data.get('IsOver')
    
    divisions = data.get('divisions', [])
    division_ids = [
        d.get('divisionId') for d in divisions if 'divisionId' in d
    ]
    
    occurred = "Yes" if division_ids else "No"
    
    return {
        "event_id": event_id,
        "event_name": event_name,
        "occurred": occurred,
        "divisions": division_ids
    }

In [ ]:
def process_event(event_id: int) -> Dict:
    """
    Fetch + parse + convert a single event.
    Always returns a structured record.
    """
    data = fetch_event(event_id)
    
    if data:
        parsed = parse_event(data)
    else:
        parsed = {
            "event_id": event_id,
            "event_name": "",
            "occurred": "No",
            "divisions": []
        }
    
    parsed["converted_id"] = numeric_to_url_id(event_id)
    
    return parsed

In [ ]:
def process_event_range(start_id: int, end_id: int, delay: float = 0.3) -> List[Dict]:
    """
    Loop through a range of event IDs and collect results.
    """
    results = []
    
    for event_id in range(start_id, end_id + 1):
        print(f"Processing event {event_id}...")
        
        result = process_event(event_id)
        results.append(result)
        
        time.sleep(delay)
    
    return results

In [ ]:
def results_to_dataframe(results: List[Dict]) -> pd.DataFrame:
    """
    Convert results list into a pandas DataFrame.
    """
    df = pd.DataFrame(results)
    
    # Convert list → string for CSV compatibility
    df['divisions'] = df['divisions'].apply(
        lambda x: ",".join(map(str, x)) if isinstance(x, list) else ""
    )
    
    return df

In [ ]:
def save_results_to_csv(df: pd.DataFrame, filename: str) -> None:
    """
    Save DataFrame to CSV.
    """
    df.to_csv(filename, index=False)

In [ ]:
#from aes_events import process_event_range, results_to_dataframe, save_results_to_csv

results = process_event_range(40000, 45000)

df = results_to_dataframe(results)

save_results_to_csv(df, "events.csv")

In [ ]:
df

In [ ]:
fetch_event(41091,10)